## 🧠 fs03 — CLIP ViT-H/14 Embedding Extraction

This notebook extracts **1280-dim CLIP ViT-H/14 LAION2B embeddings** from:

```dataset_raw/part4_sd21/pretrained_features/pretrained_features/*.pickle```


These embeddings power the attractiveness regression model.

In [10]:
import os
import pickle
import json
import numpy as np
from tqdm import tqdm

RAW_FEAT_DIR = "dataset_raw/part4_sd21/pretrained_features/pretrained_features"

OUT_FEAT_PATH = "embeddings/features.npy"
OUT_INDEX_PATH = "embeddings/feature_index.json"

os.makedirs("embeddings", exist_ok=True)

print("Embedding source:", RAW_FEAT_DIR)


Embedding source: dataset_raw/part4_sd21/pretrained_features/pretrained_features


## 📁 Scan for all `.pickle` CLIP embedding files

In [11]:
files = sorted([
    os.path.join(RAW_FEAT_DIR, f)
    for f in os.listdir(RAW_FEAT_DIR)
    if f.endswith(".pickle")
])

print("Found pickle files:", len(files))
print(files[:5])

Found pickle files: 125754
['dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000001.pickle', 'dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000002.pickle', 'dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000003.pickle', 'dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000004.pickle', 'dataset_raw/part4_sd21/pretrained_features/pretrained_features/SFHQ_pt4_00000005.pickle']


## 🧠 Extract the best-performing embedding:

### **CLIP_ViT_H_14_LAION2B**

Each `.pickle` contains multiple embeddings:

- `CLIP_ViTL_14@336`
- `CLIP_ResNet50x64`
- `CLIP_ViT_H_14_LAION2B` ← **we want this one**
- `ConvNext_XL_Imagenet21k`

This notebook loads only the ViT-H/14 vector.

In [12]:
clip_vectors = []
index_list = []

for path in tqdm(files):
    with open(path, "rb") as f:
        obj = pickle.load(f)

    if "CLIP_ViT_H_14_LAION2B" not in obj:
        print("WARNING: Missing CLIP_ViT_H_14_LAION2B in", path)
        continue

    vec = obj["CLIP_ViT_H_14_LAION2B"]

    # vec shape expected: (1, 1280)
    vec = np.asarray(vec).reshape(-1)   # → 1D (1280,)

    clip_vectors.append(vec)

    image_id = os.path.splitext(os.path.basename(path))[0]
    index_list.append({
        "id": image_id,
        "path": path
    })

clip_vectors = np.vstack(clip_vectors)
clip_vectors.shape

100%|██████████| 125754/125754 [00:19<00:00, 6496.67it/s]


(125754, 1280)

## 🔄 Normalize Embeddings

This ensures all vectors are on the same scale  
and improves downstream learning performance.

In [13]:
# L2 normalize
norms = np.linalg.norm(clip_vectors, axis=1, keepdims=True)
clip_vectors = clip_vectors / norms

clip_vectors.shape, norms.shape

((125754, 1280), (125754, 1))

## 💾 Save Embedding Matrix + Index

In [14]:
np.save(OUT_FEAT_PATH, clip_vectors)

with open(OUT_INDEX_PATH, "w") as f:
    json.dump(index_list, f, indent=2)

print("Saved:", OUT_FEAT_PATH)
print("Saved:", OUT_INDEX_PATH)
print("Final embedding matrix:", clip_vectors.shape)

Saved: embeddings/features.npy
Saved: embeddings/feature_index.json
Final embedding matrix: (125754, 1280)


## 🎉 Extraction Complete

You now have:

- `embeddings/features.npy` → shape `(125754, 1280)`
- `embeddings/feature_index.json`

You can now proceed to:

### 👉 `fs04_attractiveness_model.ipynb`